In [ ]:
suppressMessages(library(readxl))
suppressMessages(library(tidyverse))
suppressMessages(library(edgeR))
suppressMessages(library(ggpubr))
suppressMessages(library(reshape2))
suppressMessages(library(utils))
suppressMessages(library(openxlsx))
suppressMessages(library(ggfortify))
suppressMessages(library(RColorBrewer))
suppressMessages(library(ggrepel))
suppressMessages(library(ggvenn))
suppressMessages(library(clusterProfiler))
options(repr.plot.width = 10, repr.plot.height = 10)  

In [ ]:
meta <- read_excel("../metadata_X1_SCOP-2024-0277.xlsx", sheet = 1)[-c(5, 69),] %>%
  rename_with(~ "Anatomy", .cols = 10) %>%
  rename_with(~ "Sample_ID", .cols = 4) %>%
  mutate(Identifier = paste0(Condition1, "_", Anatomy, ".", `Wet lab ID`))
meta$Tissue <- ifelse(grepl("Hypothalamus", meta$Identifier), "Hypothalamus", "DVC")
meta <- janitor::clean_names(meta)

data <- read.table("/projects/perslab/people/jmg776/ad_hoc/christoffer_multi-drug/salmon.merged.gene_counts.tsv", sep = "\t", header = TRUE) %>%
  select(-gene_id) %>%
  filter(!duplicated(gene_name)) %>%
  column_to_rownames("gene_name") %>%
  round()
colnames(data) <- meta$identifier

data_hypo <- data[,grep("Hypo", colnames(data))]
group_hypo <- factor(sub("_.*", "", colnames(data_hypo)))
data_DVC <- data[,grep("DVC", colnames(data))]
group_DVC <- factor(sub("_.*", "", colnames(data_DVC)))

# Format data

In [ ]:
hypo_dge <- DGEList(counts = data_hypo, group = group_hypo)
DVC_dge <- DGEList(counts = data_DVC, group = group_DVC)

In [ ]:
# Filter genes for lowly expressed. Removes genes with less than n counts in k samples (check doc. for defaults)
hypo_dge <- hypo_dge[filterByExpr(hypo_dge), , keep.lib.sizes=FALSE]
DVC_dge <- DVC_dge[filterByExpr(DVC_dge), , keep.lib.sizes=FALSE]

In [ ]:
# Calculate normalization factors (TMM)
hypo_dge <- normLibSizes(hypo_dge)
DVC_dge <- normLibSizes(DVC_dge)

# MDS

In [ ]:
options(repr.plot.height = 15, repr.plot.width = 15)
my_colors <- c(
  "Vehicle" = "grey",
  "Cagrilinitide" = "#1b9e77", # Dark green
  "Retatrutide" = "#d95f02", # Orange
  "Combo" = "#7570b3"       # Purple
)

plotMDS(hypo_dge, col = my_colors[as.character(hypo_dge$samples$group)]) # Use pch = 16 for cicles instead of text
plotMDS(DVC_dge, col = my_colors[as.character(DVC_dge$samples$group)])

# DEG analysis

In [ ]:
# Setup design matrix
design_hypo <- model.matrix(~0 + group, data = hypo_dge$samples)

design_DVC <- model.matrix(~0 + group, data = DVC_dge$samples)

colnames(design_hypo)[1:4] <- levels(hypo_dge$samples$group)
colnames(design_DVC)[1:4]  <- levels(DVC_dge$samples$group)

## Hypothalamus

In [ ]:
hypo_contrasts <- makeContrasts(Cagri = Cagrilinitide - Vehicle,
                                Reta = Retatrutide - Vehicle,
                                Combo = Combo - Vehicle,
                                Interaction = (Combo - Cagrilinitide) - (Retatrutide - Vehicle),
                                levels = design_hypo)

v_hypo <- voomWithQualityWeights(hypo_dge, design_hypo, plot = TRUE) # Weighting samples, outliers should get lower weight
fit_hypo <- lmFit(v_hypo, design_hypo)
fit_hypo <- contrasts.fit(fit_hypo, hypo_contrasts)
fit_hypo <- eBayes(fit_hypo, robust = TRUE)

### Tables

In [ ]:
hypo_DEGs_cag_sign <- topTable(fit_hypo, coef = "Cagri", n = Inf) %>%
    #filter(adj.P.Val < 0.05) %>%
    rownames_to_column("gene") %>%
    mutate(
    treatment = "Cagrilintide",
    region = "Hypothalamus"
    )
hypo_DEGs_cag_sign

hypo_DEGs_reta_sign <- topTable(fit_hypo, coef = "Reta", n = Inf) %>%
    #filter(adj.P.Val < 0.05) %>%
    rownames_to_column("gene") %>%
    mutate(
    treatment = "Retatrutide",
    region = "Hypothalamus"
    )
hypo_DEGs_reta_sign

hypo_DEGs_combo_sign <- topTable(fit_hypo, coef = "Combo", n = Inf) %>%
    #filter(adj.P.Val < 0.05) %>%
    rownames_to_column("gene") %>%
    mutate(
    treatment = "Combo",
    region = "Hypothalamus"
    )
hypo_DEGs_combo_sign

In [ ]:
write.xlsx(rbind(hypo_DEGs_cag_sign, hypo_DEGs_reta_sign, hypo_DEGs_combo_sign), file = "hypo_DEGs_significant.xlsx")

### Visualizations

In [ ]:
venn_data_hypo <- list(
  `Cagri vs. Vehicle` = rownames(hypo_DEGs_cag_sign %>% filter(adj.P.Val < 0.05)),
  `Reta vs. Vehicle` = rownames(hypo_DEGs_reta_sign %>% filter(adj.P.Val < 0.05)),
  `Combo vs. Vehicle` = rownames(hypo_DEGs_combo_sign %>% filter(adj.P.Val < 0.05))
)

ggvenn(
  venn_data_hypo,
  stroke_size = 1,
  set_name_size = 6) +
  ggtitle("Hypothalamus") + theme(plot.title = element_text(size = 24, face = "bold", hjust = 0.5, margin = margin(b = 20))
)

In [ ]:
library(ggrepel)

plot_df <- hypo_DEGs_combo_sign %>%
  mutate(
    sig = adj.P.Val < 0.01,
    regulation = case_when(
      sig & logFC > 1 ~ "Upregulated",
      sig & logFC < -1 ~ "Downregulated",
      TRUE            ~ "Not significant"
    ),
    label = ifelse(sig & abs(logFC) > 1, gene, NA_character_)
  )

up_color   <- "#8f1d1e"
down_color <- "#1e90ff"

# Volcano plot
p_volcano <- ggplot(plot_df, aes(x = logFC, y = -log10(adj.P.Val))) +
  geom_point(aes(color = regulation), size = 1.5, alpha = 0.7) +
  scale_color_manual(values = c(
    "Upregulated"     = up_color,
    "Downregulated"   = down_color,
    "Not significant" = "grey70"
  )) +
  geom_hline(yintercept = -log10(0.05), linetype = "dashed", color = "black") +
  geom_text_repel(
    aes(label = label, color = regulation),
    size = 2,
    box.padding = 0.3,
    segment.size = 0.3,
    max.overlaps = 20,
    show.legend = FALSE
  ) +
  theme_classic(base_size = 16) +
  theme(
    legend.position = "none",
    axis.title = element_text(size = 6),
    axis.text  = element_text(size = 6),
    axis.line  = element_line(color = "black", linewidth = 0.1),
    panel.border = element_rect(color = "black", fill = NA, linewidth = 1.5),
    plot.title = element_blank()
  ) +
  labs(
    x = bquote(log[2]~Fold~Change),
    y = expression(-log[10]~P-value)
  )

p_volcano

ggsave("Volcano_Hypo_Combo.pdf", height = 5, width = 6, units = "cm", dpi = 600)

## DVC

In [ ]:
DVC_contrasts <- makeContrasts(Cagri = Cagrilinitide - Vehicle,
                                Reta = Retatrutide - Vehicle,
                                Combo = Combo - Vehicle,
                                Interaction = (Combo - Cagrilinitide) - (Retatrutide - Vehicle),
                                levels = design_DVC)

v_DVC <- voomWithQualityWeights(DVC_dge, design_DVC, plot = TRUE) # Weighting samples, outliers should get lower weight
fit_DVC <- lmFit(v_DVC, design_DVC)
fit_DVC <- contrasts.fit(fit_DVC, DVC_contrasts)
fit_DVC <- eBayes(fit_DVC, robust = TRUE)

### Tables

In [ ]:
DVC_DEGs_cag_sign <- topTable(fit_DVC, coef = "Cagri", n = Inf) %>%
    filter(adj.P.Val < 0.05) %>%
    rownames_to_column("gene") %>%
    mutate(
    treatment = "Cagrilintide",
    region = "DVC"
    )
DVC_DEGs_cag_sign

DVC_DEGs_reta_sign <- topTable(fit_DVC, coef = "Reta", n = Inf) %>%
    filter(adj.P.Val < 0.05) %>%
    rownames_to_column("gene") %>%
    mutate(
    treatment = "Retatrutide",
    region = "DVC"
    )
DVC_DEGs_reta_sign

DVC_DEGs_combo_sign <- topTable(fit_DVC, coef = "Combo", n = Inf) %>%
    filter(adj.P.Val < 0.05) %>%
    rownames_to_column("gene") %>%
    mutate(
    treatment = "Combo",
    region = "DVC"
    )
DVC_DEGs_combo_sign

In [ ]:
write.xlsx(rbind(DVC_DEGs_cag_sign, DVC_DEGs_reta_sign, DVC_DEGs_combo_sign), file = "DVC_DEGs_significant.xlsx")

### Visualization

In [ ]:
venn_data_dvc <- list(
  `Vehicle vs. Cagri` = rownames(DVC_DEGs_cag_sign %>% filter(adj.P.Val < 0.05)),
  `Vehicle vs. Reta` = rownames(DVC_DEGs_reta_sign %>% filter(adj.P.Val < 0.05)),
  `Vehicle vs. Combo` = rownames(DVC_DEGs_combo_sign %>% filter(adj.P.Val < 0.05))
)

ggvenn(
  venn_data_dvc,
  stroke_size = 1,
  set_name_size = 6) + ggtitle("DVC") + theme(plot.title = element_text(size = 24, face = "bold", hjust = 0.5, margin = margin(b = 20))
)
# ggsave("Venn_DVC.png", dpi = 600)

In [ ]:
library(ggrepel)

plot_df <- DVC_DEGs_combo_sign %>%
  mutate(
    sig = adj.P.Val < 0.01,
    regulation = case_when(
      sig & logFC > 1 ~ "Upregulated",
      sig & logFC < -1 ~ "Downregulated",
      TRUE            ~ "Not significant"
    ),
    label = ifelse(sig & abs(logFC) > 1, gene, NA_character_)
  )

up_color   <- "#8f1d1e"
down_color <- "#1e90ff"

# Volcano plot
p_volcano <- ggplot(plot_df, aes(x = logFC, y = -log10(adj.P.Val))) +
  geom_point(aes(color = regulation), size = 1.5, alpha = 0.7) +
  scale_color_manual(values = c(
    "Upregulated"     = up_color,
    "Downregulated"   = down_color,
    "Not significant" = "grey70"
  )) +
  geom_hline(yintercept = -log10(0.05), linetype = "dashed", color = "black") +
  geom_text_repel(
    aes(label = label, color = regulation),
    size = 2,
    box.padding = 0.3,
    segment.size = 0.3,
    max.overlaps = 30,
    show.legend = FALSE
  ) +
  theme_classic(base_size = 16) +
  theme(
    legend.position = "none",
    axis.title = element_text(size = 6),
    axis.text  = element_text(size = 6),
    axis.line  = element_line(color = "black", linewidth = 0.1),
    panel.border = element_rect(color = "black", fill = NA, linewidth = 1.5),
    plot.title = element_blank()
  ) +
  labs(
    x = bquote(log[2]~Fold~Change),
    y = expression(-log[10]~P-value)
  )

p_volcano

ggsave("Volcano_DVC_Combo.pdf", height = 5, width = 6, units = "cm", dpi = 600)